# AI-Powered Telecom Customer Churn Analytics
**Author:** Praneet Verma

Explore customer churn and train AI classifiers using IBM's public Telco sample. The notebook downloads the CSV at runtime. Install `requirements.txt`, open in Jupyter, and run top to bottom.

Dataset: [IBM CSV](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv). Predictions are exploratory associations, not causal explanations or guaranteed outcomes.

## 1. Imports and reproducibility

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
from sklearn.inspection import permutation_importance
RANDOM_STATE=42; sns.set_theme(style="whitegrid")


## 2. Load and inspect data

In [ ]:
DATA_URL="https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_PATH=None  # Set a local CSV path for offline execution.
df=pd.read_csv(DATA_PATH or DATA_URL)
print(df.shape); display(df.head()); display(df.isna().sum().sort_values(ascending=False).head(10)); display(df['Churn'].value_counts())


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4)); sns.countplot(data=df,x='Churn',ax=ax[0]); ax[0].set_title('Churn counts')
sns.countplot(data=df,x='Contract',hue='Churn',ax=ax[1]); ax[1].tick_params(axis='x',rotation=15); ax[1].set_title('Churn by contract'); plt.tight_layout(); plt.show()


## 3. Clean and prepare data
`customerID` is excluded. Median/mode imputation, scaling, and one-hot encoding are fitted in pipelines after the stratified split to reduce leakage.

In [ ]:
clean=df.copy(); clean['TotalCharges']=pd.to_numeric(clean['TotalCharges'],errors='coerce')
y=clean['Churn'].astype(str).str.strip().map({'Yes':1,'No':0})
X=clean.drop(columns=['Churn','customerID'],errors='ignore')
X=X.drop(columns=[c for c in X if X[c].nunique(dropna=False)<=1])
num=X.select_dtypes(include=np.number).columns.tolist(); cat=[c for c in X if c not in num]
try: enc=OneHotEncoder(handle_unknown='ignore',sparse_output=True)
except TypeError: enc=OneHotEncoder(handle_unknown='ignore',sparse=True)
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('onehot',enc)]),cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
print('Rows:',len(X),'Churn prevalence:',round(y.mean(),3))


## 4. Train and compare AI models
ROC-AUC and average precision summarize ranking. Threshold metrics below use the illustrative 0.50 cutoff.

In [ ]:
models={'Logistic Regression':Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=42))]),'Random Forest':Pipeline([('prep',prep),('model',RandomForestClassifier(n_estimators=350,min_samples_leaf=3,class_weight='balanced',n_jobs=-1,random_state=42))])}
rows=[]
for name,m in models.items():
 m.fit(X_train,y_train); p=m.predict_proba(X_test)[:,1]; pred=(p>=.5).astype(int)
 rows.append({'Model':name,'ROC-AUC':roc_auc_score(y_test,p),'Average precision':average_precision_score(y_test,p),'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0)})
results=pd.DataFrame(rows).set_index('Model').sort_values('ROC-AUC',ascending=False); display(results.round(3)); best_name=results.index[0]; best=models[best_name]; print('Selected:',best_name)


## 5. Evaluation and interpretation

In [ ]:
prob=best.predict_proba(X_test)[:,1]; pred=(prob>=.5).astype(int)
print(classification_report(y_test,pred,target_names=['Stayed','Churned'],zero_division=0))
fig,ax=plt.subplots(1,3,figsize=(15,4)); ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Stayed','Churned'],ax=ax[0],colorbar=False); RocCurveDisplay.from_predictions(y_test,prob,ax=ax[1]); PrecisionRecallDisplay.from_predictions(y_test,prob,ax=ax[2]); plt.tight_layout(); plt.show()
perm=permutation_importance(best,X_test,y_test,scoring='roc_auc',n_repeats=8,random_state=42,n_jobs=-1)
imp=pd.Series(perm.importances_mean,index=X_test.columns).sort_values(ascending=False); display(imp.head(12).to_frame('Mean ROC-AUC decrease'))
imp.head(12).sort_values().plot.barh(figsize=(8,5)); plt.title('Permutation importance'); plt.tight_layout(); plt.show()


## 6. Business interpretation and limitations
Use segment patterns and model diagnostics to form retention hypotheses, then validate interventions with controlled experiments and cost-benefit analysis. Permutation importance is not causal evidence; correlated features can share importance. The sample may not represent a current provider or population. A random holdout does not establish future-period performance. Consider threshold costs, temporal/external validation, fairness, privacy, calibration, and drift before operational use. Do not treat individual model scores as certain outcomes.

## Conclusion
This notebook covers exploratory analysis, preprocessing, model comparison, evaluation, and interpretation. Run all cells and record actual metrics in the accompanying report; no performance is assumed in advance.